<img src="images/smalagent.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;">We're going to study another framework called `smolagents`, developed by Hugging Face. It allows you to generate Python code that will support reasoning (thus benefiting from all the Python modules already developed to interface with the external world).

You can still pay a few euros (this time per month) to run the agents on Hugging Face's servers, but you can also use existing servers like `ragarenn` and `ollama` locally.

In [69]:
from smolagents import OpenAIServerModel, CodeAgent, InferenceClientModel
from dotenv import load_dotenv
import os
load_dotenv(override=True)

RAGARENN_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/ch@t/api/"

rennes_model = OpenAIServerModel(
    model_id="mistralai/Mistral-Small-3.1-24B-Instruct-2503",
    api_base=RAGARENN_BASE_URL,   # your OpenAI-compatible base URL
    api_key=os.environ["RENNES_API_KEY"],
    temperature=0.1,
    max_tokens=2048,
    flatten_messages_as_text=True,  # key workaround for picky servers
)

ollama_model = OpenAIServerModel(
    model_id="qwen3:8b",  # This can be any name, LM Studio will use whatever model you have loaded
    api_base="http://localhost:11434/v1",  # Default LM Studio API endpoint
    api_key="not-needed",  # LM Studio doesn't require an API key by default
)

# Hugging Face model, HF_API_KEY must be defined in your environment variables
# Access is limited in the Free version
hf_model = InferenceClientModel(
    model_id="meta-llama/Llama-3.3-70B-Instruct",
    api_key=os.environ["HF_TOKEN"],
    )

We'll just display the system prompt of the created agent to understand the instructions.

In [40]:
agent = CodeAgent(
    name="My_test_agent",
    model=ollama_model,
    tools=[],
    verbosity_level=2
)
print(agent.system_prompt)

You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples 

We can see that the instructions given to the agent are quite simple: it must reason using Python and standard modules. It can also use external tools if they are available.

Now it's up to us to make it work by asking it a simple question.

In [41]:
response = agent.run("Combien vaut fibbonacci(10) ?")

╭──────────────────────────────────────────── New run - My_test_agent ────────────────────────────────────────────╮
│                                                                                                                 │
│ Combien vaut fibbonacci(10) ?                                                                                   │
│                                                                                                                 │
╰─ OpenAIServerModel - qwen3:8b ──────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, let's tackle the problem "Combien vaut fibbonacci(10) ?" which translates to "What is the value of           
fibonacci(10)?"                                                                                                    
                                                                                                                   
First, I need to compute the 10th Fibonacci number. The Fibonacci sequence starts with 0 and 1, and each subsequent
number is the sum of the two preceding ones. So the sequence goes 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, etc.            
                                                                                                                   
Wait, sometimes the sequence starts with 1 and 1 instead of 0 and 1. I should confirm which definition is being    
used here. But since the question is in French, maybe the standard definition applies. Let me think: the 10th term.
Let's list them out:                                                                                               
                                                                                                                   
If starting from 0:                                                                                                
Term 1: 0                                                                                                          
Term 2: 1                                                                                                          
Term 3: 1 (0+1)                                                                                                    
Term 4: 2 (1+1)                                                                                                    
Term 5: 3 (1+2)                                                                                                    
Term 6: 5 (2+3)                                                                                                    
Term 7: 8 (3+5)                                                                                                    
Term 8: 13 (5+8)                                                                                                   
Term 9: 21 (8+13)                                                                                                  
Term 10: 34 (13+21)                                                                                                
                                                                                                                   
So the 10th term would be 34. Alternatively, if the count starts at 1 as the first term, then term 10 is 34.       
                                                                                                                   
Alternatively, maybe the [0

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  def fibonacci(n):                                                                                                
      if n == 1:                                                                                                   
          return 0                                                                                                 
      elif n == 2:                                                                                                 
          return 1                                                                                                 
      a, b = 0, 1                                                                                                  
      for _ in range(3, n+1):                                                                                      
          a, b = b, a + b                                                                                          
      return b                                                                                                     
                                                                                                                   
  result = fibonacci(10)                                                                                           
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 34

[Step 1: Duration 103.45 seconds| Input tokens: 2,036 | Output tokens: 1,526]

<img src="images/pop.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> We can also ask for more interesting things:

The agent goes into three steps:
* thought about the problem
* wrote code to compute the answer and execute it
* make an observation of the result, if the result is not satisfactory, it can iterate again.
Finally it returns the final answer.

Let's do it with some more valuable queries:

In [42]:
agent.run("What is the latest news about Taylor Swift?")

╭──────────────────────────────────────────── New run - My_test_agent ────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest news about Taylor Swift?                                                                     │
│                                                                                                                 │
╰─ OpenAIServerModel - qwen3:8b ──────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, the user is asking for the latest news about Taylor Swift. Let me think about how to approach this.          
                                                                                                                   
First, I need to figure out what tools I have available. The tools mentioned are Python functions, and the user has
provided examples using functions like web_search and final_answer. Since there's no specific tool listed here, but
in the previous examples, web_search was used for similar tasks, I'll assume that web_search is the appropriate    
tool here.                                                                                                         
                                                                                                                   
So, the plan is to use the web_search function to look up the latest news about Taylor Swift. The query should be  
straightforward: "Taylor Swift latest news". I'll structure the code to perform this search and then print the     
results.                                                                                                           
                                                                                                                   
Wait, the user might have multiple news sources, so I should make sure to capture the most recent information.     
Maybe the web_search function can handle that by returning the latest results. Once I get the search results, I can
present them as the answer.                                                                                        
                                                                                                                   
I should also check if there are any other tools needed, but based on the examples, web_search seems sufficient.   
Let me write the code to 

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tags and that the final answer is provided using the final_answer tool. Let me test this approach.               
  </think>                                                                                                         
                                                                                                                   
  Thought: I will use the `web_search` tool to find the latest news about Taylor Swift.                            
  <code>                                                                                                           
  latest_news = web_search(query="Taylor Swift latest news")                                                       
  print(latest_news)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code parsing failed on line 1 due to: SyntaxError
tags and that the final answer is provided using the final_answer tool. Let me test this approach.
               ^
Error: invalid syntax (<unknown>, line 1)

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, the user is asking for the latest news about Taylor Swift. Let me try to figure out how to approach this.    
                                                                                                                   
First, I remember that in the previous attempts, the assistant tried using the `web_search` tool, but there was an 
error because that tool wasn't allowed. Oh right, the available tools are limited to certain Python functions, and 
`web_search` isn't among them. Wait, the user mentioned that the only tool available is `final_answer`, but that's 
for providing the final answer. Hmm, maybe there's a misunderstanding here.                                        
                                                                                                                   
Wait, looking back at the tools provided, the user only has access to the `final_answer` tool, which is for giving 
the final answer. But the previous examples used other tools like `document_qa`, `image_generator`, `translator`,  
`image_qa`, `wikipedia_search`, and `web_search`. However, in this case, the user might not have access to those   
tools. Wait, no—the user's initial message says that the assistant has access to a list of tools, but in the       
current task, the assistant is supposed to use only the allowed tools. But in the current scenario, the assistant  
is getting an error because `web_search` isn't allowed.                                                            
                                                                                                                   
Wait, [38;2;230;2

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("I cannot retrieve real-time news. Check official sources or recent media outlets for Taylor        
  Swift's latest updates.")                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: I cannot retrieve real-time news. Check official sources or recent media outlets for Taylor Swift's 
latest updates.

[Step 3: Duration 49.17 seconds| Input tokens: 6,774 | Output tokens: 1,454]

"I cannot retrieve real-time news. Check official sources or recent media outlets for Taylor Swift's latest updates."

It doesn't work, our agent tried to call the functions `web_search` or `wikipedia_search` which are not known to the Python interpreter. We need to reconfigure our agent.

We can see that we have to allow the use of the `json`module to manipulate the requested JSON output.

In [44]:
from smolagents import WebSearchTool, DuckDuckGoSearchTool

agent = CodeAgent(
    name="My_test_agent2",
    model=ollama_model,
    tools=[DuckDuckGoSearchTool()],
    additional_authorized_imports=["json"],
    add_base_tools=True,  # adds Python REPL, file system access, and more
    verbosity_level=2
)

result = agent.run("""What is the latest 5 news articles about Taylor Swift?
          Give the result as a JSON list containing a short summary and a link to the article.""")

print("Réponse finale:")
print(result)

╭─────────────────────────────────────────── New run - My_test_agent2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest 5 news articles about Taylor Swift?                                                          │
│           Give the result as a JSON list containing a short summary and a link to the article.                  │
│                                                                                                                 │
╰─ OpenAIServerModel - qwen3:8b ──────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, I need to find the latest 5 news articles about Taylor Swift and present them as a JSON list with summaries  
and links. Let's think about how to approach this.                                                                 
                                                                                                                   
First, I should use the web_search tool to get recent articles. The query should be something like "latest Taylor  
Swift news articles" or similar. But wait, the web_search function returns the top results as a string. Maybe I can
search for "Taylor Swift latest news" and then extract the top 5 results. However, the tool's output might not be  
structured, so I need to handle that.                                                                              
                                                                                                                   
Wait, the user wants a JSON list. So perhaps I can perform the search, get the links, then for each link, visit the
webpage to extract the summary. But the visit_webpage function might not be necessary if the search results already
include the summaries. Alternatively, maybe the web_search returns the top links, and I can parse those. Let me    
check the tools again.                                                                                             
                                                                                                                   
The web_search function returns the top search results as a string. The visit_webpage function visits a URL and    
returns the content as a markdown string. So, perhaps first, I [38;2;23

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = "Taylor Swift latest news articles"                                                                      
  search_results = web_search(query=query)                                                                         
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Taylor Swift live updates: News, photos, fan theories, 
more](https://pagesix.com/2025/10/23/entertainment/taylor-swift-live-updates-10-23-24-2025/)
Follow our Taylor Swift live updates for the latest on her new album, peeks into her relationship with fiancé 
Travis Kelce, plus news , photos, fan theories and more.

[Taylor Swift talks falling in love with Travis Kelce, making ... - ABC 
News](https://abcnews.go.com/GMA/Culture/taylor-swift-talks-falling-love-travis-kelce-making/story?id=126249372)
Taylor Swift discussed her new album, her love for Travis Kelce and more in a recent interview.

[Taylor Swift's new album breaks her own sales records - BBC](https://www.bbc.com/news/articles/ckg28el7r4po)
Taylor Swift's latest album, The Life of a Showgirl, has already secured the UK's biggest opening week of 2025, 
after selling 304,000 copies since Friday. The total eclipses the first-week sales ...

[Taylor Swift | Latest News | New York Post](https://nypost.com/tag/taylor-swift/)
Get the latest taylor swift news , articles , videos and photos on the New York Post.

[Taylor Swift - The New York Times](https://www.nytimes.com/spotlight/taylor-swift)
After the release of her latest album, an aquarium in California and a museum in Germany suddenly were thrust into 
the center of Taylor Swift's fandom. Just how big is Swift's cultural reach?

[Why fans and critics alike are slamming Taylor Swift's new 
album](https://www.npr.org/2025/10/10/nx-s1-5566389/why-fans-and-critics-alike-are-slamming-taylor-swifts-new-album
)
Culture Why fans and critics alike are slamming Taylor Swift's new album October 10, 20254:47 PM ET Heard on All 
Things Considered By

[Taylor Swift releases "The Life of a Showgirl" to much 
fanfare](https://www.cbsnews.com/news/taylor-swift-the-life-of-a-showgirl-album-release/)
Taylor Swift's 12th studio album, "The Life of a Showgirl," was released at midnight Friday. Swift wrote and 
produced the album with Swedish producers Max Martin and Shellback.

[Taylor Swift | AP News](https://apnews.com/hub/taylor-swift)
Stay informed and read the latest breaking news and updates on Taylor Swift from AP News , the definitive source 
for independent journalism.

[Taylor Swift - BBC News](https://www.bbc.co.uk/news/topics/c5elz99kyrgt)
Taylor Swift - get the latest news , video and top stories about the singer-songwriter as well as updates on tour 
dates and tickets, from experts across the BBC.

[Taylor Swift on 'New Heights': Biggest Reveals Not About Her New 
Album](https://www.billboard.com/lists/taylor-swift-new-heights-biggest-reveals/)
Taylor Swift's 'New Heights' episode featured major takeaways beyond news about her new album. Here are the biggest
reveals.

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<think>                                                                                                            
Okay, I need to find the latest 5 news articles about Taylor Swift and present them as a JSON list with summaries  
and links. Let me start by using the web_search tool to get recent articles. The user provided some search results,
so I should check those.                                                                                           
                                                                                                                   
Looking at the observation, there are several articles listed. I need to pick the top 5. Each entry has a title and
a snippet. I'll extract the title as the summary and the link. Wait, the first result has a title and a            
description. Let me make sure to get the correct summary from the snippet. For example, the first article mentions 
her new album and fan theories, so the summary could be "Taylor Swift live updates: News, photos, fan theories,    
more" with the link. Similarly, the second article's summary is about her talking about Travis Kelce and the album.
I'll need to format each of these into a JSON object with "summary" and "link" fields. Let me check if there are 5 
articles in the observation. The first 5 entries are from pagesix, abcnews, bbc, nypost, and nytimes. That's five. 
I'll take those. Now, structure them into[38;2;230;237;243

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import json                                                                                                      
                                                                                                                   
  articles = [                                                                                                     
      {                                                                                                            
          "summary": "Taylor Swift live updates: News, photos, fan theories, more",                                
          "link": "https://pagesix.com/2025/10/23/entertainment/taylor-swift-live-updates-10-23-24-2025/"          
      },                                                                                                           
      {                                                                                                            
          "summary": "Taylor Swift talks falling in love with Travis Kelce, making ...",                           
          "link":                                                                                                  
  "https://abcnews.go.com/GMA/Culture/taylor-swift-talks-falling-love-travis-kelce-making/story?id=126249372"      
      },                                                                                                           
      {                                                                                                            
          "summary": "Taylor Swift's new album breaks her own sales records",                                      
          "link": "https://www.bbc.com/news/articles/ckg28el7r4po"                                                 
      },                                                                                                           
      {                                                                                                            
          "summary": "Taylor Swift | Latest News | New York Post",                                                 
          "link": "https://nypost.com/tag/taylor-swift/"                                                           
      },                                                                                                           
      {                                                                                                            
          "summary": "Taylor Swift - The New York Times",                                                          
          "link": "https://www.nytimes.com/spotlight/taylor-swift"                                                 
      }                                                                                                            
  ]                                                                                                                
                                                                                                                   
  final_answer(json.dumps(articles, indent=2))                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: [
  {
    "summary": "Taylor Swift live updates: News, photos, fan theories, more",
    "link": "https://pagesix.com/2025/10/23/entertainment/taylor-swift-live-updates-10-23-24-2025/"
  },
  {
    "summary": "Taylor Swift talks falling in love with Travis Kelce, making ...",
    "link": 
"https://abcnews.go.com/GMA/Culture/taylor-swift-talks-falling-love-travis-kelce-making/story?id=126249372"
  },
  {
    "summary": "Taylor Swift's new album breaks her own sales records",
    "link": "https://www.bbc.com/news/articles/ckg28el7r4po"
  },
  {
    "summary": "Taylor Swift | Latest News | New York Post",
    "link": "https://nypost.com/tag/taylor-swift/"
  },
  {
    "summary": "Taylor Swift - The New York Times",
    "link": "https://www.nytimes.com/spotlight/taylor-swift"
  }
]

[Step 2: Duration 38.45 seconds| Input tokens: 5,054 | Output tokens: 1,791]

Réponse finale:
[
  {
    "summary": "Taylor Swift live updates: News, photos, fan theories, more",
    "link": "https://pagesix.com/2025/10/23/entertainment/taylor-swift-live-updates-10-23-24-2025/"
  },
  {
    "summary": "Taylor Swift talks falling in love with Travis Kelce, making ...",
    "link": "https://abcnews.go.com/GMA/Culture/taylor-swift-talks-falling-love-travis-kelce-making/story?id=126249372"
  },
  {
    "summary": "Taylor Swift's new album breaks her own sales records",
    "link": "https://www.bbc.com/news/articles/ckg28el7r4po"
  },
  {
    "summary": "Taylor Swift | Latest News | New York Post",
    "link": "https://nypost.com/tag/taylor-swift/"
  },
  {
    "summary": "Taylor Swift - The New York Times",
    "link": "https://www.nytimes.com/spotlight/taylor-swift"
  }
]


<img src="images/concert.png" width="150" alt="Several agents" style="float: left; margin-right: 15px; margin-bottom: 10px;"> We would like to know when Taylor Swift is touring in France, we can query ticketmaster website, using 

In [53]:
from smolagents import CodeAgent

# Set your Hugging Face API token

agent = CodeAgent(
    tools=[], 
    model=rennes_model,
    additional_authorized_imports=
    ["selenium", 
    "selenium.webdriver.common.by",
    "selenium.webdriver.common.keys", "json"],
    verbosity_level=2
)

# Prompt with updated details for a local browser and retries
prompt = """
What's lemonde.fr is reporting about Taylor Swift.
"""

# Run the prompt
agent.run(prompt)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What's lemonde.fr is reporting about Taylor Swift.                                                              │
│                                                                                                                 │
╰─ OpenAIServerModel - mistralai/Mistral-Small-3.1-24B-Instruct-2503 ─────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
'NoneType' object is not subscriptable

[Step 1: Duration 0.22 seconds]

AgentGenerationError: Error in generating model output:
'NoneType' object is not subscriptable

In [70]:
agent.run("hello")



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ hello                                                                                                           │
│                                                                                                                 │
╰─ OpenAIServerModel - mistralai/Mistral-Small-3.1-24B-Instruct-2503 ─────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Error code: 401 - {'detail': 'Your session has expired or the token is invalid. Please sign in again.'}

[Step 1: Duration 0.03 seconds]

AgentGenerationError: Error in generating model output:
Error code: 401 - {'detail': 'Your session has expired or the token is invalid. Please sign in again.'}